# exp063_ravaghi_vs_pixiux_lgbm_feature_parity_audit inference

Strict public replay inference for the selected Pixiux likelihood-PF LightGBM candidate. This notebook loads the saved public LightGBM fold boosters from the train notebook output, generates only test-side replay features, averages saved booster predictions, and writes `submission.csv` without hidden-specific branches or visible override logic.

## Contents

1. Setup and configuration
2. Raw competition input check
3. Public replay inference
4. Submission and artifacts

## 1. Setup and configuration

In [ ]:
from pathlib import Path
import json
import pandas as pd

from settings import ExperimentPaths, load_config, get_nested
from public_notebook_replay_audit import run_public_replay_inference

def cfg_get(config, dotted_key, default=None):
    value = get_nested(config, dotted_key)
    return default if value is None else value

In [ ]:
paths = ExperimentPaths()
paths.ensure_output_dirs()
config = load_config()

print("Experiment:", config["experiment"]["name"])
print("Route:", config["experiment"]["route"])
print("Mode:", cfg_get(config, "inference.mode", "strict_public_replay_saved_model_inference_no_override"))
print("Variant:", cfg_get(config, "inference.variant", "pixiux_likpf_public_replay"))
print("Model:", cfg_get(config, "inference.model", "lgb_mean"))
print("Submission path:", paths.submission_path)

## 2. Raw competition input check

In [ ]:
train_dir = paths.train_data_dir
test_dir = paths.test_data_dir
sample_path = paths.sample_submission_path
train_files = sorted(train_dir.glob("*__horizontal_well.csv"))
test_files = sorted(test_dir.glob("*__horizontal_well.csv"))
print("Train dir:", train_dir, "wells=", len(train_files))
print("Test dir:", test_dir, "wells=", len(test_files))
print("Sample submission:", sample_path, "exists=", sample_path.exists())
if not train_files:
    raise FileNotFoundError(f"No train wells found under {train_dir}")
if not test_files:
    raise FileNotFoundError(f"No test wells found under {test_dir}")
display(pd.read_csv(sample_path, nrows=5))

## 3. Public replay inference

In [ ]:
summary = run_public_replay_inference(
    data_dir=paths.raw_data_dir,
    output_dir=paths.artifacts_dir,
    submission_path=paths.submission_path,
    n_jobs=cfg_get(config, "runtime.num_workers", 8),
    pf_seeds=cfg_get(config, "audit.pf_seeds", 128),
    pf_particles=cfg_get(config, "audit.pf_particles", 500),
    fast=bool(cfg_get(config, "audit.fast", False)),
    use_gpu=str(cfg_get(config, "audit.use_gpu", "auto")),
    max_wells=cfg_get(config, "audit.max_wells"),
    variant=cfg_get(config, "inference.variant", "pixiux_likpf_public_replay"),
    model_name=cfg_get(config, "inference.model", "lgb_mean"),
    model_artifact_dir=cfg_get(config, "inference.model_artifact_dir"),
    sample_submission_path=paths.sample_submission_path,
    submission_target_column=cfg_get(config, "data.submission_target_column", "tvt"),
)
print(json.dumps(summary, indent=2))

## 4. Submission and artifacts

In [ ]:
metrics = pd.read_csv(paths.artifacts_dir / "ravaghi_vs_pixiux_public_replay_inference_metrics.csv")
schema = pd.read_csv(paths.artifacts_dir / "ravaghi_vs_pixiux_public_replay_inference_feature_schema.csv")
test_predictions = pd.read_csv(paths.artifacts_dir / "ravaghi_vs_pixiux_public_replay_inference_test_predictions.csv.gz")
submission = pd.read_csv(paths.submission_path)
tracker_test_path = paths.artifacts_dir / "ravaghi_vs_pixiux_public_replay_tracker_features_test.csv.gz"

display(metrics)
display(schema.groupby("variant").size().rename("feature_count").reset_index())
display(test_predictions.groupby(["variant", "model"]).size().rename("rows").reset_index())
display(submission.head())
print("Submission rows:", len(submission))
print("Submission path:", paths.submission_path)
print("Reusable tracker test features:", tracker_test_path, "exists=", tracker_test_path.exists())
print("Fallback rows:", summary["submission"]["fallback_rows"])